# FORELÆSNING 9: Lineær regression med populationsdata

> Lineær regression på populationsniveau, modelvalidering, residualanalyse og forudsigelse af kontinuerte udfald.

### Underviser: Martin Siemienski Andersen mvan@hst.aau.dk

**ST2 - Anvendt Programmering**

# ST2 ANVENDT PROGRAMMERING - Overblik

## Alle forelaesninger


In [ ]:
from IPython.display import HTML
import requests

url = "https://raw.githubusercontent.com/AAU-ST2-Programming/all_lectures/refs/heads/main/overview_files/shared_overview_table9.html"
html = requests.get(url, timeout=30).text
HTML(html)

# Læringsmål

Efter dagens forelæsning kan du:

**Forstå lineær regression på populationsniveau**
- Anvende regression til at forudsige kontinuerte udfald fra features
- Fortolke hældning, skæring og R²

**Forberede og rense populationsdata**
- Håndtere manglende data og outliers
- Organisere data som NumPy arrays

**Evaluere regressionsmodeller**
- Beregne R², RMSE og residualer
- Validere modelantagelser visuelt

**Implementere multiple lineær regression**
- Bruge flere features som prædiktorer
- Forstå multikollinearitet

**Sikre reproducerbarhed**
- Dokumentere analyse med metadata

# Fra signaler til populationsdata

## Forskel på signalniveau og populationsniveau

| Aspekt | Signalniveau (Forelæsning 5-8) | Populationsniveau (Forelæsning 9-12) |
|--------|-------------------------------|--------------------------------------|
| **Data** | Tidsserier (EKG, PPG, SCG) | Tværsnitsdata (én måling per person) |
| **Struktur** | 1D array (samples over tid) | 2D array (rækker=personer, kolonner=features) |
| **Regression** | Forudsige beat-features fra signalkarakteristika | Forudsige helbredsudfald fra personkarakteristika |
| **Visualisering** | Linjeplot (tid på x-aksen) | Scatter plot (ingen tidsdimension) |
| **Eksempel** | HR fra PPG-amplitude | Blodtryk fra alder og BMI |

# lad os starte med at se på noget data, omkring normale hjertefrekvenser i en population


In [ ]:
import numpy as np

# Load the data using numpy with structured array
data = np.genfromtxt('files/synthetic_hr_data.csv', delimiter=',', skip_header=1, 
                      dtype='float,float,float', names='age,gender,heart_rate')

# Extract columns
age = data['age']
gender = data['gender']  # 0 = Female, 1 = Male
heart_rate = data['heart_rate']


# Separate heart rates by gender
females = heart_rate[gender == 0]
males = heart_rate[gender == 1]


In [ ]:

import matplotlib.pyplot as plt
# Plot boxplot of heart rates by gender
plt.figure(figsize=(10, 6))
plt.boxplot([females, males], tick_labels=['Female', 'Male'], vert=True)
plt.title('Boxplot of Heart Rates by Gender')
plt.xlabel('Gender')
plt.ylabel('Heart Rate (bpm)')
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Create visualizations
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: Heart Rate vs Age (colored by gender)
females = data[data['gender'] == 0]
males = data[data['gender'] == 1]
axes[0, 0].scatter(females['age'], females['heart_rate'], alpha=0.4, s=10, color='red', label='Female')
axes[0, 0].scatter(males['age'], males['heart_rate'], alpha=0.4, s=10, color='blue', label='Male')
axes[0, 0].set_xlabel('Age (years)')
axes[0, 0].set_ylabel('Heart Rate (bpm)')
axes[0, 0].set_title('Heart Rate vs Age (by Gender)')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Plot 2: Heart Rate distribution by gender
axes[0, 1].hist(females['heart_rate'], bins=30, alpha=0.6, label='Female', color='red', edgecolor='black')
axes[0, 1].hist(males['heart_rate'], bins=30, alpha=0.6, label='Male', color='blue', edgecolor='black')
axes[0, 1].set_xlabel('Heart Rate (bpm)')
axes[0, 1].set_ylabel('Frequency')
axes[0, 1].set_title('Distribution of Heart Rate by Gender')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3, axis='y')

# Plot 3: Box plot by gender
axes[1, 0].boxplot([females['heart_rate'].values, males['heart_rate'].values], 
                    tick_labels=['Female (0)', 'Male (1)'])
axes[1, 0].set_xlabel('Gender')
axes[1, 0].set_ylabel('Heart Rate (bpm)')
axes[1, 0].set_title('Heart Rate Distribution by Gender')

# Plot 4: Age distribution
axes[1, 1].hist(data['age'], bins=40, edgecolor='black', alpha=0.7, color='green')
axes[1, 1].set_xlabel('Age (years)')
axes[1, 1].set_ylabel('Frequency')
axes[1, 1].set_title('Distribution of Ages')
axes[1, 1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## Hvorfor populationsregression?
- Identificer **risikofaktorer** (hvilke features forudsiger sygdom?)
- **Forudsig udfald** for nye individer
- Forstå **sammenhænge** mellem karakteristika

# Hvad er lineær regression?

## Definition
Lineær regression finder den **rette linje** der bedst beskriver sammenhængen mellem:
- **X**: prediktor/feature (uafhængig variabel)
- **y**: respons/udfald (afhængig variabel)

## Ligningen
$$y = mx + b$$

hvor:
- $m$ = **hældning** (hvor meget $y$ ændrer sig per enhed $X$)
- $b$ = **skæring** (værdien af $y$ når $X = 0$)

## Metode: Mindste kvadraters metode (Ordinary Least Squares)
Modellen finder $m$ og $b$ ved at **minimere kvadrerede fejl** (sum of squared residuals):

$$\text{SSE} = \sum_{i=1}^{n} (y_i - \hat{y}_i)^2 = \sum_{i=1}^{n} (y_i - (mx_i + b))^2$$

hvor $\hat{y}_i$ er den forudsagte værdi for observation $i$.

hvor $\bar{x}$ og $\bar{y}$ er middelværdierne af $X$ og $y$.

### Matematisk løsning

Ved at differentiere SSE med hensyn til $m$ og $b$ og sætte lig nul, får vi:$$b = \bar{y} - m\bar{x}$$


$$m = \frac{\sum_{i=1}^{n}(x_i - \bar{x})(y_i - \bar{y})}{\sum_{i=1}^{n}(x_i - \bar{x})^2}$$

In [ ]:
# Manuel implementering af lineær regression
import numpy as np
import matplotlib.pyplot as plt

# Data: [alder (år), blodtryk (mmHg)] - flere datapunkter
x = np.array([25, 30, 35, 40, 45, 50, 55, 60, 65, 70, 75])
y = np.array([110, 115, 118, 122, 128, 135, 140, 145, 150, 155, 160])

print("DATA:")
print(f"x (alder):    {x}")
print(f"y (blodtryk): {y}")
plt.scatter(x,y)
plt.xlabel("Alder")
plt.ylabel("Blodtryk")
plt.show()

## Trin 1: Beregn middelværdier

In [ ]:

# Trin 1: Beregn middelværdier
x_mean = np.mean(x)
y_mean = np.mean(y)

print("\nTRIN 1: MIDDELVÆRDIER")
print(f"x̄ (gennemsnitsalder):    {x_mean:.1f} år")
print(f"ȳ (gennemsnitsblodtryk): {y_mean:.1f} mmHg")



## Trin 2: Beregn afvigelser fra middelværdi

In [ ]:
# Trin 2: Beregn afvigelser fra middelværdi
x_diff = x - x_mean
y_diff = y - y_mean

print("\nTRIN 2: AFVIGELSER FRA MIDDELVÆRDI")
print(f"(x - x̄): {x_diff}")
print(f"(y - ȳ): {y_diff}")

## Trin 3: Beregn hældning (m)

In [ ]:

# Trin 3: Beregn hældning (m)
numerator = np.sum(x_diff * y_diff)
denominator = np.sum(x_diff ** 2)
m = numerator / denominator

print("\nTRIN 3: BEREGN HÆLDNING (m)")
print(f"Tæller:   Σ(x-x̄)(y-ȳ) = {numerator:.1f}")
print(f"Nævner:   Σ(x-x̄)²      = {denominator:.1f}")
print(f"Hældning: m = {m:.4f} mmHg/år")

## Trin 4: Beregn skæring (b)

In [ ]:

# Trin 4: Beregn skæring (b)
b = y_mean - m * x_mean

print("\nTRIN 4: BEREGN SKÆRING (b)")
print(f"b = ȳ - m·x̄")
print(f"b = {y_mean:.1f} - {m:.4f}·{x_mean:.1f}")
print(f"b = {b:.2f} mmHg")

# Resultat

In [ ]:

# Resultat
print("\n" + "=" * 50)
print("RESULTAT")
print("=" * 50)
print(f"Regressionslinje: y = {m:.4f}x + {b:.2f}")
print(f"\nFortolkning:")
print(f"  - For hvert år ældre stiger blodtrykket {m:.4f} mmHg")
print(f"  - Ved alder 0 ville BT være {b:.2f} mmHg (ekstrapolation)")
print("=" * 50)

# Beregn forudsagte værdier
y_pred = m * x + b

# Plot data og regressionslinje
plt.figure(figsize=(10, 6))

# Scatter plot af faktiske data
plt.scatter(x, y, color='blue', s=100, alpha=0.7, label='Faktiske data', zorder=3)

# Regressionslinje
plt.plot(x, y_pred, color='red', linewidth=2.5, label=f'Regression: y = {m:.2f}x + {b:.1f}', zorder=2)

# Tilføj grid bag datapunkterne
plt.grid(True, alpha=0.3, linestyle='--', zorder=1)

# Labels og titel
plt.xlabel('Alder (år)', fontsize=12, fontweight='bold')
plt.ylabel('Systolisk blodtryk (mmHg)', fontsize=12, fontweight='bold')
plt.title('Lineær regression: Blodtryk vs Alder', fontsize=14, fontweight='bold', pad=15)

# Legend
plt.legend(fontsize=11, loc='lower right')

# Tilpas layout
plt.tight_layout()

# Gem figur
plt.savefig('regression_plot.png', dpi=300, bbox_inches='tight')
print("\nPlot gemt som 'regression_plot.png'")

# Vis plot
plt.show()

print("\nBemærk:")
print("  - Blå punkter viser de faktiske målinger")
print("  - Rød linje viser den beregnede regressionslinje")
print("  - Jo tættere punkterne er på linjen, jo bedre er fittet")

# PRAKTISK IMPLEMENTERING

## Trin 1: Forbered data

Vi skal have data i den rigtige form for regression.

In [ ]:
# Import nødvendige biblioteker
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression

# Eksempel: Forudsig systolisk blodtryk fra alder
# Data: [alder (år), systolisk blodtryk (mmHg)]
data = np.array([
    [25, 110], [30, 115], [35, 118], [40, 122], 
    [45, 128], [50, 135], [55, 140], [60, 145],
    [65, 150], [70, 155], [75, 160]
])

# Split i X (prediktor) og y (respons)
X = data[:, 0:1] # Alder (skal være 2D for sklearn), tænk [[features for patient 0], [features for patient 1...]]
y = data[:, 1]   # Blodtryk (1D array)

print(f"X shape: {X.shape}")  # (11, 1) - 11 personer, 1 feature
print(f"y shape: {y.shape}")  # (11,) - 11 målinger
print(f"\nFørste 3 datapunkter:")
print(f"Alder: {X[:3].flatten()}")
print(f"Blodtryk: {y[:3]}")

## Trin 2: Fit regressionsmodellen

Nu fitter vi modellen til data ved at finde den bedste linje.

In [ ]:
# Opret og fit model
model = LinearRegression()
model.fit(X, y)

# Udtræk parametre
m = model.coef_[0]      # Hældning
b = model.intercept_    # Skæring

print("=" * 50)
print("REGRESSIONSPARAMETRE")
print("=" * 50)
print(f"Hældning (m):     {m:.3f} mmHg/år")
print(f"Skæring (b):      {b:.3f} mmHg")
print(f"\nModelformel:  y = {m:.3f}x + {b:.3f}")
print("=" * 50)

# Fortolkning
print(f"\nFortolkning:")
print(f"   - For hvert år ældre stiger blodtrykket {m:.3f} mmHg")
print(f"   - Ved alder 0 (ekstrapolation) ville BT være {b:.3f} mmHg")

## Trin 3: Lav forudsigelser

Brug modellen til at forudsige nye værdier.

In [ ]:
# Forudsigelser for træningsdata
y_pred = model.predict(X)

# Forudsigelser for nye aldere
alder_ny = np.array([[32], [48], [63]])  # 3 nye personer
blodtryk_forudsagt = model.predict(alder_ny)

print("FORUDSIGELSER FOR NYE PERSONER")
print("=" * 50)
for alder, bt in zip(alder_ny.flatten(), blodtryk_forudsagt):
    print(f"Alder {alder} år → Forudsagt BT: {bt:.1f} mmHg")
print("=" * 50)

# Sammenlign faktiske vs forudsagte (træningsdata)
print(f"\nFørste 3 træningspunkter:")
print(f"Faktisk BT:    {y[:3]}")
print(f"Forudsagt BT:  {y_pred[:3].round(1)}")

## Trin 4: Visualisér regressionen

Visualisering viser hvor godt linjen fitter data.

In [ ]:
# Plot data og regressionslinje
plt.figure(figsize=(10, 6))

# Scatter plot af data
plt.scatter(X, y, color='blue', s=100, alpha=0.6, label='Faktiske data')

# Regressionslinje
plt.plot(X, y_pred, color='red', linewidth=2, label=f'Regression: y = {m:.2f}x + {b:.2f}')

# Tilpasning
plt.xlabel('Alder (år)', fontsize=12)
plt.ylabel('Systolisk blodtryk (mmHg)', fontsize=12)
plt.title('Lineær regression: Blodtryk vs Alder', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Bemærk:")
print("   - Blå punkter = faktiske målinger")
print("   - Rød linje = den fittede model")
print("   - Jo tættere punkterne er på linjen, jo bedre fit")

# Modelevaluering

## Hvor godt er fittet?

Vi bruger tre hovedmetrikker:

### 1. **R² (R-squared)**
- Andel af varians forklaret af modellen
- Interval: 0 til 1 (0 = ingen forklaring, 1 = perfekt fit)
- $R^2 = 1 - \frac{\sum(y_i - \hat{y}_i)^2}{\sum(y_i - \bar{y})^2}$

### 2. **RMSE (Root Mean Squared Error)**
- Gennemsnitlig afstand mellem forudsigelser og data
- Samme enhed som y (mmHg i vores eksempel)
- $\text{RMSE} = \sqrt{\frac{1}{n}\sum(y_i - \hat{y}_i)^2}$

### 3. **Residualer**
- Forskellen mellem faktisk og forudsagt: $\text{residual}_i = y_i - \hat{y}_i$
- Bør være tilfældige (ingen mønster)

In [ ]:
# Beregn evalueringsmetrikker
r2 = model.score(X, y)  # R² fra sklearn
rmse = np.sqrt(np.mean((y - y_pred)**2))  # RMSE
residuals = y - y_pred  # Residualer

print("=" * 50)
print("MODELEVALUERING")
print("=" * 50)
print(f"R² (R-squared):        {r2:.4f}")
print(f"RMSE:                  {rmse:.2f} mmHg")
print(f"Mean residual:         {np.mean(residuals):.2e} mmHg")
print(f"Std. dev. residuals:   {np.std(residuals):.2f} mmHg")
print("=" * 50)

# Fortolkning
print(f"\nFortolkning:")
print(f"   - R² = {r2:.4f} → Modellen forklarer {r2*100:.1f}% af variansen")
if r2 > 0.9:
    print(f"   - Dette er et MEGET GODT fit!")
elif r2 > 0.7:
    print(f"   - Dette er et GODT fit")
else:
    print(f"   - Dette fit kunne forbedres")
print(f"   - RMSE = {rmse:.2f} mmHg → Typisk forudsigelsefejl")
print(f"   - Residualer har mean ≈ 0 (som forventet)")

# Residualanalyse

## Hvorfor residualer?

Residualer fortæller os:
- **Hvor godt modellen fitter** (små residualer = godt fit)
- **Om modellen er korrekt** (tilfældig spredning = godt, mønster = problem)
- **Om antagelser holder** (normalitet, homoskedasticitet)

## God vs Dårlig residualfordeling

| God residualfordeling | Dårlig residualfordeling |
|-----------------------|--------------------------|
| Tilfældig spredning omkring 0 | Systematisk mønster (kurve, vifteform) |
| Konstant varians | Variansen stiger/falder med X |
| Omtrent normalfordelt | Skæv fordeling, outliers |

In [ ]:
# Visualisér residualer
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Residualer vs Forudsagt
axes[0].scatter(y_pred, residuals, color='purple', s=100, alpha=0.6)
axes[0].axhline(y=0, color='red', linestyle='--', linewidth=2, label='y=0')
axes[0].set_xlabel('Forudsagt blodtryk (mmHg)', fontsize=12)
axes[0].set_ylabel('Residualer (mmHg)', fontsize=12)
axes[0].set_title('Residualer vs Forudsagt', fontsize=13, fontweight='bold')
axes[0].grid(True, alpha=0.3)
axes[0].legend()

# Plot 2: Histogram af residualer
axes[1].hist(residuals, bins=8, color='green', alpha=0.7, edgecolor='black')
axes[1].axvline(x=0, color='red', linestyle='--', linewidth=2, label='Mean=0')
axes[1].set_xlabel('Residualer (mmHg)', fontsize=12)
axes[1].set_ylabel('Frekvens', fontsize=12)
axes[1].set_title('Fordeling af residualer', fontsize=13, fontweight='bold')
axes[1].grid(True, alpha=0.3, axis='y')
axes[1].legend()

plt.tight_layout()
plt.show()

print("Interpretation:")
print("   - Venstre plot: Tilfældig spredning omkring 0 = godt")
print("   - Højre plot: Omtrent symmetrisk fordeling = godt")
print("   - Hvis der var et mønster, skulle vi bruge en ikke-lineær model")

# Multiple lineær regression

## Fra én til flere features

**Simpel lineær regression:**
$$y = m x + b$$
- Én prediktor (X)

**Multiple lineær regression:**
$$y = m_1 x_1 + m_2 x_2 + m_3 x_3 + ... + m_n x_n + b$$
- Flere prædiktorer ($x_1, x_2, ..., x_n$)

## Eksempel: Blodtryk fra alder OG BMI
$$\text{Blodtryk} = m_1 \cdot \text{Alder} + m_2 \cdot \text{BMI} + b$$

## Fordele
- Mere præcise forudsigelser (højere R²)  
- Forstå relative bidrag fra forskellige faktorer  
- Kontrollere for confounders  

## Udfordringer
- Multikollinearitet (korrelerede features)  
- Overfitting med for mange features  
- Fortolkning bliver mere kompleks

## Multiple regression: Implementering

In [ ]:
# Data med flere features: [alder, BMI, systolisk BT]
data_multi = np.array([
    [25, 22, 110], [30, 24, 115], [35, 23, 118], [40, 26, 122], 
    [45, 28, 128], [50, 30, 135], [55, 29, 140], [60, 31, 145],
    [65, 32, 150], [70, 33, 155], [75, 35, 160]
])

# Split i X (flere features) og y
X_multi = data_multi[:, 0:2]  # Alder og BMI (2 kolonner)
y_multi = data_multi[:, 2]     # Blodtryk

print(f"X_multi shape: {X_multi.shape}")  # (11, 2) - 11 personer, 2 features
print(f"y_multi shape: {y_multi.shape}")  # (11,) - 11 målinger

In [ ]:
# Fit multiple regression model
model_multi = LinearRegression()
model_multi.fit(X_multi, y_multi)

# Udtræk parametre
m1, m2 = model_multi.coef_
b_multi = model_multi.intercept_
print("\n" + "=" * 50)
print("MULTIPLE REGRESSION RESULTAT")
print("=" * 50)
print(f"Koefficient for Alder (m1): {m1:.3f} mmHg/år")
print(f"Koefficient for BMI (m2):   {m2:.3f} mmHg/(kg/m²)")
print(f"Skæring (b):                {b_multi:.3f} mmHg")
print(f"\nModelformel:")
print(f"BT = {m1:.3f}·Alder + {m2:.3f}·BMI + {b_multi:.3f}")
print("=" * 50)


In [ ]:
# Evaluer
y_pred_multi = model_multi.predict(X_multi)
r2_multi = model_multi.score(X_multi, y_multi)
rmse_multi = np.sqrt(np.mean((y_multi - y_pred_multi)**2))

print(f"\nEvaluering:")
print(f"R²:   {r2_multi:.4f} (sammenlign med simple model: {r2:.4f})")
print(f"RMSE: {rmse_multi:.2f} mmHg (sammenlign med simple model: {rmse:.2f} mmHg)")

print(f"\nMultiple regression forbedrer R² med {(r2_multi-r2)*100:.1f}%!... hmm det var ikke meget.")

# Datahåndtering: Manglende data og outliers

## Manglende data (NaN)

**Problem:** Reelle datasæt indeholder ofte manglende værdier

**Løsninger:**
1. **Fjern rækker:** Drop personer med manglende data
   ```python
   # Identificér rækker uden NaN
   mask = ~np.isnan(data).any(axis=1)
   data_clean = data[mask]
   ```

2. **Imputation:** Erstat med gennemsnit, median eller forudsigelse
   ```python
   from sklearn.impute import SimpleImputer
   imputer = SimpleImputer(strategy='mean')
   data_imputed = imputer.fit_transform(data)
   ```

## Outliers

**Problem:** Ekstreme værdier kan dominere fittet

**Løsninger:**
1. **Boks visualisering:** Inspicer først med box plots!
2. **Z-score metode:** Fjern punkter > 3 standardafvigelser fra middelværdi
3. **IQR metode:** Fjern punkter uden for [Q1-1.5·IQR, Q3+1.5·IQR]

In [ ]:
# Eksempel: Data med manglende værdier og outlier
data_messy = np.array([
    [25, 110], [30, 115], [35, np.nan], [40, 122],  # NaN ved alder 35
    [45, 128], [50, 135], [55, 200], [60, 145],     # Outlier (200) ved alder 55
    [65, 150], [70, 155], [75, 160]
])

print("ORIGINAL DATA")
print(data_messy)
print(f"\nAntal NaN: {np.isnan(data_messy).sum()}")

# 1. Håndter manglende data: Fjern rækker med NaN
mask = ~np.isnan(data_messy).any(axis=1)
data_no_nan = data_messy[mask]
print(f"\nEfter fjernelse af NaN: {data_no_nan.shape[0]} rækker tilbage")

# 2. Identificér outliers (Z-score > 3)
X_clean = data_no_nan[:, 0].reshape(-1, 1)
y_clean = data_no_nan[:, 1]

z_scores = np.abs((y_clean - np.mean(y_clean)) / np.std(y_clean))
outlier_mask = z_scores < 3  # Behold kun ikke-outliers

X_final = X_clean[outlier_mask]
y_final = y_clean[outlier_mask]

print(f"Efter fjernelse af outliers: {X_final.shape[0]} rækker tilbage")
print(f"Fjernet outlier ved alder: {X_clean[~outlier_mask].flatten()}")

# Fit ren model
model_clean = LinearRegression()
model_clean.fit(X_final, y_final)
r2_clean = model_clean.score(X_final, y_final)

print(f"\nRen model R²: {r2_clean:.4f}")
print(f"   (Sammenlign med oprindelig model: {r2:.4f})")

# Reproducerbarhed og dokumentation

## Hvorfor metadata?

Uden metadata kan analysen ikke verificeres eller genbruges!

## Hvad skal dokumenteres?

### Data-metadata
- Datakilde og indsamlingsdato
- Antal personer, antal features
- Enheder for hver variabel
- Behandlinger udført (fjernet NaN? outliers?)

### Model-metadata
- Model-type (LinearRegression)
- Software-versioner (sklearn, numpy)
- Feature-navne og rækkefølge
- Fit-dato og analytiker

### Resultat-metadata
- R², RMSE og andre metrikker
- Koefficienter og skæring
- Valideringsstrategi (train/test split?)

## Gem metadata i JSON eller CSV
```python
import json
metadata = {
    "date": "2026-02-10",
    "model": "LinearRegression",
    "features": ["alder", "BMI"],
    "n_samples": 11,
    "r2": 0.9876,
    "rmse": 2.34
}
with open("metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)
```

# Komplet workflow: Fra data til model

```
┌─────────────────────────────────────────────────────────────┐
│ 1. INDLÆS DATA                                              │
│    - Læs CSV/NumPy array                                    │
│    - Tjek struktur og dimensioner                           │
└─────────────────────────────────────────────────────────────┘
                            ↓
┌─────────────────────────────────────────────────────────────┐
│ 2. INSPICÉR & RENS                                          │
│    - Visualisér med scatter plots                           │
│    - Tjek for manglende data (np.isnan)                     │
│    - Identificér og håndter outliers                        │
└─────────────────────────────────────────────────────────────┘
                            ↓
┌─────────────────────────────────────────────────────────────┐
│ 3. FORBERED                                                 │
│    - Split i X (features) og y (respons)                    │
│    - Reshape til korrekte dimensioner                       │
│    - (Optional) Normaliser/standardisér                     │
└─────────────────────────────────────────────────────────────┘
                            ↓
┌─────────────────────────────────────────────────────────────┐
│ 4. FIT MODEL                                                │
│    - LinearRegression().fit(X, y)                           │
│    - Udtræk koefficienter og skæring                        │
└─────────────────────────────────────────────────────────────┘
                            ↓
┌─────────────────────────────────────────────────────────────┐
│ 5. EVALUER                                                  │
│    - Beregn R², RMSE                                        │
│    - Plot data + regressionslinje                           │
│    - Analyser residualer                                    │
└─────────────────────────────────────────────────────────────┘
                            ↓
┌─────────────────────────────────────────────────────────────┐
│ 6. FORUDSIG & DOKUMENTÉR                                    │
│    - Brug model.predict() på nye data                       │
│    - Gem resultater med metadata                            │
│    - Rapportér resultater                                   │
└─────────────────────────────────────────────────────────────┘
```

# Antagelser og begrænsninger

## Lineær regressions antagelser

For at regression skal være valid, skal følgende holde:

### 1. **Linearitet**
- Sammenhængen mellem X og y skal være lineær
- Tjek: Plot data først!

### 2. **Uafhængighed**
- Observationer skal være uafhængige
- Problem: gentagne målinger på samme person

### 3. **Homoskedasticitet**
- Varians af residualer skal være konstant
- Tjek: Residualplot ikke-vifteformet

### 4. **Normalitet**
- Residualer bør være normalfordelte
- Tjek: Histogram eller Q-Q plot

## Begrænsninger

**Korrelation ≠ kausalitet**
- Regression viser sammenhæng, ikke årsag

**Ekstrapolering er usikker**
- Forudsigelser uden for data-området er upålidelige

**Multikollinearitet**
- Korrelerede features gør koefficienter ustabile

**Outliers dominerer**
- Få ekstreme punkter kan ændre fittet drastisk

# Praktiske tips

## Gode vaner

### **Visualisér ALTID data først**
- Scatter plot afslører relationer og outliers
- Aldrig fit en model uden at se data

### **Start simpelt**
- Prøv simple lineær regression før multiple
- Tilføj features én ad gangen

### **Validér antagelser**
- Tjek residualplots efter hvert fit
- Hvis mønster: overvej ikke-lineær model

### **Dokumentér alt**
- Gem metadata sammen med resultater
- Kommentér kode grundigt
- Rapportér både succesfulde og mislykkede forsøg

### **Fortolk i kontekst**
- R² alene er ikke nok
- Spørg: Giver resultaterne fysiologisk/klinisk mening?

## Almindelige fejl

- Glemme at reshape X til 2D array  
- Bruge træningsdata til evaluering  
- Ignorere outliers og NaN  
- Antage kausalitet fra korrelation  
- Ekstrapolere langt fra data  
- Glemme metadata (hvordan blev modellen fittetet?)

# Etiske overvejelser

## Populationsdata er følsomme helbredsdata

### Privatlivsbeskyttelse
- **Pseudonymisering:** Ingen direkte identifikatorer (navne, CPR-numre)
- **Adgangskontrol:** Kun autoriserede personer ser data
- **Kryptering:** Data skal være krypteret både i hvile og under transport

### Bias og retfærdighed
- **Repræsentativitet:** Er din population repræsentativ?
- **Underrepræsentation:** Visse grupper kan mangle i data
- **Fairness:** Sikr at modellen ikke diskriminerer på køn, etnicitet, alder

### Samtykke og transparens
- **Informeret samtykke:** Personer skal vide hvordan data bruges
- **Formålsbegrænsning:** Brug kun data til det aftalte formål
- **Retten til at blive glemt:** Personer kan trække samtykke tilbage

### Ansvarlig brug af forudsigelser
- **Usikkerhed kommunikeres:** RMSE og konfidensintervaller skal rapporteres
- **Begrænsninger ekspliciteres:** Ikke ekstrapolér, rapportér antagelser
- **Klinisk validering:** Modeller skal testes prospektivt før deployment

### Refleksion
> Når vi forudsiger helbredsudfald, påvirker vi virkelige menneskers liv.  
> Spørg altid: Er dette etisk forsvarligt? Er det til gavn for patienten?

<div style="background-color: rgba(255, 253, 231, 0.2); padding: 30px; border-radius: 10px; border-left: 5px solid #fbc02d;">

# ØVELSE: Prøv selv!

## Opgave: Forudsig diastolisk blodtryk fra puls

Du har følgende data: [puls (bpm), diastolisk blodtryk (mmHg)]

### Trin 1: Indlæs data
```python
data_exercise = np.array([
    [60, 70], [65, 72], [70, 75], [75, 78],
    [80, 80], [85, 83], [90, 85], [95, 88]
])
```

### Trin 2: Opgaver
1. **Split data** i X (puls) og y (diastolisk BT)
2. **Fit model** med `LinearRegression()`
3. **Visualisér** data + regressionslinje
4. **Evaluer** med R² og RMSE
5. **Analyser residualer** (plot residualer vs forudsagt)
6. **Forudsig** diastolisk BT ved puls = 72 bpm

### Trin 3: Udvidelse (multiple regression)
Tilføj BMI som ekstra feature og se om R² forbedres:
```python
data_extended = np.array([
    [60, 22, 70], [65, 24, 72], [70, 23, 75], [75, 26, 78],
    [80, 28, 80], [85, 30, 83], [90, 29, 85], [95, 31, 88]
])
# Kolonner: puls, BMI, diastolisk BT
```

### Forventede resultater
- R² bør være > 0.90
- Residualer bør være tilfældigt spredt
- Multiple regression bør forbedre R²

## Start her!

</div>

In [ ]:
# DIN KODE HER: Løs øvelsen ovenfor

# Trin 1: Indlæs data
data_exercise = np.array([
    [60, 70], [65, 72], [70, 75], [75, 78],
    [80, 80], [85, 83], [90, 85], [95, 88]
])

# Trin 2: Din kode...

# Reference: sklearn funktioner

## Vigtigste funktioner til lineær regression

### **Model-oprettelse og fitting**
```python
from sklearn.linear_model import LinearRegression

model = LinearRegression()
model.fit(X, y)  # X skal være 2D, y skal være 1D
```

### **Udtræk parametre**
```python
m = model.coef_         # Hældning(er) - array
b = model.intercept_    # Skæring - scalar
```

### **Forudsigelser**
```python
y_pred = model.predict(X)         # Forudsig for X
y_new = model.predict(X_new)      # Forudsig for nye data
```

### **Evaluering**
```python
r2 = model.score(X, y)            # R² score
```

### **Andre nyttige værktøjer**
```python
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Standardisér features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Håndter manglende data
imputer = SimpleImputer(strategy='mean')
X_imputed = imputer.fit_transform(X)
```

### **NumPy helpers**
```python
# Fjern NaN
mask = ~np.isnan(data).any(axis=1)
data_clean = data[mask]

# Beregn RMSE
rmse = np.sqrt(np.mean((y - y_pred)**2))

# Beregn residualer
residuals = y - y_pred
```

# Tjekliste: Lineær regression

Brug denne tjekliste hver gang du laver regression:

## Før modellering

- [ ] Data indlæst og inspiceret (shape, type, range)
- [ ] Scatter plot lavet for at se relationer
- [ ] Manglende data identificeret og håndteret
- [ ] Outliers identificeret og beslutning taget
- [ ] X er 2D array, y er 1D array
- [ ] Features og respons er klart defineret

## Under modellering

- [ ] Model oprettet: `LinearRegression()`
- [ ] Model fittet: `model.fit(X, y)`
- [ ] Koefficienter udtrukket: `model.coef_`, `model.intercept_`
- [ ] Forudsigelser lavet: `y_pred = model.predict(X)`

## Efter modellering

- [ ] R² beregnet og fortolket
- [ ] RMSE beregnet og fortolket
- [ ] Residualer beregnet og plottet
- [ ] Residualplot ser tilfældigt ud (ingen mønster)
- [ ] Data + regressionslinje visualiseret
- [ ] Resultater giver fysiologisk/klinisk mening
- [ ] Metadata dokumenteret og gemt
- [ ] Antagelser tjekket (linearitet, homoskedasticitet)

## Etik og privatliv

- [ ] Data er pseudonymiseret
- [ ] Adgangskontrol implementeret
- [ ] Samtykke indhentet og respekteret
- [ ] Usikkerhed kommunikeret (RMSE, CI)
- [ ] Begrænsninger ekspliciteret


In [ ]:
from IPython.display import HTML
import requests

url = "https://raw.githubusercontent.com/AAU-ST2-Programming/all_lectures/refs/heads/main/overview_files/shared_overview_table.html"
html = requests.get(url, timeout=30).text
HTML(html)